# 06_sol: SQL + Python Data Cleaning

Contains:
- the same scenario as `06_mock`
- one complete reference implementation
- grading tests


In [ ]:
# Chunk overview: Prepare imports, fixtures, and helper scaffolding used by the solution.
# Why this chunk exists: it makes the solution easier to reason about under interview time pressure.
# Import required modules for this solution step.
import sqlite3
# Import required modules for this solution step.
from collections import defaultdict
# Import required modules for this solution step.
from datetime import datetime
# Import required modules for this solution step.
from typing import Any

# Assign computed data to a named variable for later use.
RAW_SALES = [
    # Execute this line as part of the solution flow.
    ("o1", "2025-01-02", "us", "$1,200.00", "2025-01-02T10:00:00"),
    # Execute this line as part of the solution flow.
    ("o1", "2025-01-02", "US", "$1,250.00", "2025-01-02T12:00:00"),  # latest wins
    # Execute this line as part of the solution flow.
    ("o2", "01/03/2025", " eu ", "850", "2025-01-03T09:00:00"),
    # Execute this line as part of the solution flow.
    ("o3", "2025-01-03", "", "N/A", "2025-01-03T09:30:00"),  # invalid amount -> drop
    # Execute this line as part of the solution flow.
    ("o4", "2025-01-04", "apac", "300.5", "2025-01-04T08:00:00"),
    # Execute this line as part of the solution flow.
    ("o5", "2025-13-04", "us", "100", "2025-01-04T08:00:00"),  # invalid date -> drop
    # Execute this line as part of the solution flow.
    ("o6", "2025-01-04", None, " 99.50 ", "2025-01-04T10:00:00"),
# Execute this line as part of the solution flow.
]


# Define `make_connection` so this step is reusable and testable.
def make_connection() -> sqlite3.Connection:
    # Assign computed data to a named variable for later use.
    conn = sqlite3.connect(":memory:")
    # Execute this line as part of the solution flow.
    conn.execute(
        # Execute this line as part of the solution flow.
        """
        # Execute this line as part of the solution flow.
        CREATE TABLE sales_raw (
            # Execute this line as part of the solution flow.
            order_id TEXT NOT NULL,
            # Execute this line as part of the solution flow.
            order_date TEXT NOT NULL,
            # Execute this line as part of the solution flow.
            region TEXT,
            # Execute this line as part of the solution flow.
            amount_text TEXT NOT NULL,
            # Execute this line as part of the solution flow.
            updated_at TEXT NOT NULL
        # Call this function to perform the next operation.
        )
        # Execute this line as part of the solution flow.
        """
    # Call this function to perform the next operation.
    )
    # Execute this line as part of the solution flow.
    conn.executemany(
        # Execute this line as part of the solution flow.
        "INSERT INTO sales_raw (order_id, order_date, region, amount_text, updated_at) VALUES (?, ?, ?, ?, ?)",
        # Execute this line as part of the solution flow.
        RAW_SALES,
    # Call this function to perform the next operation.
    )
    # Call this function to perform the next operation.
    conn.commit()
    # Return the computed value for the caller.
    return conn


In [ ]:
# Chunk overview: Implement the final reference solution in a clean, stepwise way.
# Why this chunk exists: it makes the solution easier to reason about under interview time pressure.
# Define `parse_amount` so this step is reusable and testable.
def parse_amount(amount_text: str) -> float | None:
    # Assign computed data to a named variable for later use.
    raw = amount_text.strip().replace("$", "").replace(",", "")
    # Check this condition to choose the correct branch.
    if raw in {"", "N/A", "n/a", "NULL", "null"}:
        # Return the computed value for the caller.
        return None
    # Start guarded block to handle potential runtime errors.
    try:
        # Return the computed value for the caller.
        return float(raw)
    # Handle expected failure path and keep behavior predictable.
    except ValueError:
        # Return the computed value for the caller.
        return None


# Define `parse_order_date` so this step is reusable and testable.
def parse_order_date(raw_date: str) -> str | None:
    # Iterate through items to process each element deterministically.
    for fmt in ("%Y-%m-%d", "%m/%d/%Y"):
        # Start guarded block to handle potential runtime errors.
        try:
            # Return the computed value for the caller.
            return datetime.strptime(raw_date, fmt).strftime("%Y-%m-%d")
        # Handle expected failure path and keep behavior predictable.
        except ValueError:
            # Execute this line as part of the solution flow.
            continue
    # Return the computed value for the caller.
    return None


# Define `extract_clean_rows` so this step is reusable and testable.
def extract_clean_rows(conn: sqlite3.Connection) -> list[dict[str, Any]]:
    # Assign computed data to a named variable for later use.
    sql = """
        # Execute this line as part of the solution flow.
        SELECT s.order_id, s.order_date, s.region, s.amount_text, s.updated_at
        # Execute this line as part of the solution flow.
        FROM sales_raw s
        # Execute this line as part of the solution flow.
        JOIN (
            # Execute this line as part of the solution flow.
            SELECT order_id, MAX(updated_at) AS max_updated_at
            # Execute this line as part of the solution flow.
            FROM sales_raw
            # Execute this line as part of the solution flow.
            GROUP BY order_id
        # Execute this line as part of the solution flow.
        ) latest
        # Assign computed data to a named variable for later use.
        ON s.order_id = latest.order_id AND s.updated_at = latest.max_updated_at
        # Execute this line as part of the solution flow.
        ORDER BY s.order_id
    # Execute this line as part of the solution flow.
    """
    # Assign computed data to a named variable for later use.
    clean: list[dict[str, Any]] = []
    # Iterate through items to process each element deterministically.
    for order_id, order_date, region, amount_text, _updated_at in conn.execute(sql):
        # Assign computed data to a named variable for later use.
        amount = parse_amount(amount_text)
        # Assign computed data to a named variable for later use.
        date_iso = parse_order_date(order_date)
        # Assign computed data to a named variable for later use.
        region_norm = (region or "").strip().upper() or "UNKNOWN"
        # Check this condition to choose the correct branch.
        if amount is None or date_iso is None or amount <= 0:
            # Execute this line as part of the solution flow.
            continue
        # Execute this line as part of the solution flow.
        clean.append(
            # Execute this line as part of the solution flow.
            {
                # Execute this line as part of the solution flow.
                "order_id": order_id,
                # Execute this line as part of the solution flow.
                "order_date": date_iso,
                # Execute this line as part of the solution flow.
                "region": region_norm,
                # Execute this line as part of the solution flow.
                "amount": amount,
            # Execute this line as part of the solution flow.
            }
        # Call this function to perform the next operation.
        )
    # Return the computed value for the caller.
    return clean


# Define `summarize_by_region` so this step is reusable and testable.
def summarize_by_region(rows: list[dict[str, Any]]) -> dict[str, float]:
    # Assign computed data to a named variable for later use.
    totals: defaultdict[str, float] = defaultdict(float)
    # Iterate through items to process each element deterministically.
    for row in rows:
        # Assign computed data to a named variable for later use.
        totals[row["region"]] += float(row["amount"])
    # Return the computed value for the caller.
    return {region: round(total, 2) for region, total in sorted(totals.items())}


# Define `top_day` so this step is reusable and testable.
def top_day(rows: list[dict[str, Any]]) -> tuple[str, float]:
    # Assign computed data to a named variable for later use.
    day_totals: defaultdict[str, float] = defaultdict(float)
    # Iterate through items to process each element deterministically.
    for row in rows:
        # Assign computed data to a named variable for later use.
        day_totals[row["order_date"]] += float(row["amount"])
    # Check this condition to choose the correct branch.
    if not day_totals:
        # Raise explicit error to fail fast on invalid state.
        raise ValueError("no_rows")
    # Return the computed value for the caller.
    return sorted(day_totals.items(), key=lambda kv: (-kv[1], kv[0]))[0]


In [ ]:
# Chunk overview: Run checks that prove the implementation meets the problem contract.
# Why this chunk exists: it makes the solution easier to reason about under interview time pressure.
# Define `run_exam06_tests` so this step is reusable and testable.
def run_exam06_tests() -> None:
    # Assign computed data to a named variable for later use.
    conn = make_connection()
    # Assign computed data to a named variable for later use.
    rows = extract_clean_rows(conn)

    # Assert expected behavior to validate correctness.
    assert [row["order_id"] for row in rows] == ["o1", "o2", "o4", "o6"]
    # Assert expected behavior to validate correctness.
    assert rows[0]["amount"] == 1250.0  # latest o1 row wins
    # Assert expected behavior to validate correctness.
    assert rows[1]["order_date"] == "2025-01-03"
    # Assert expected behavior to validate correctness.
    assert rows[3]["region"] == "UNKNOWN"

    # Assign computed data to a named variable for later use.
    summary = summarize_by_region(rows)
    # Assert expected behavior to validate correctness.
    assert summary == {
        # Execute this line as part of the solution flow.
        "APAC": 300.5,
        # Execute this line as part of the solution flow.
        "EU": 850.0,
        # Execute this line as part of the solution flow.
        "UNKNOWN": 99.5,
        # Execute this line as part of the solution flow.
        "US": 1250.0,
    # Execute this line as part of the solution flow.
    }

    # Assign computed data to a named variable for later use.
    day, total = top_day(rows)
    # Assert expected behavior to validate correctness.
    assert day == "2025-01-02"
    # Assert expected behavior to validate correctness.
    assert total == 1250.0

    # Assert expected behavior to validate correctness.
    assert parse_amount("N/A") is None
    # Assert expected behavior to validate correctness.
    assert parse_order_date("2025-13-01") is None

    # Call this function to perform the next operation.
    print("06_mock tests passed")


# Call this function to perform the next operation.
run_exam06_tests()
